Load the data

In [69]:
import pandas as pd

# Load the CSV files
matches_file_path = 'data/epic7_matches.csv'
hero_codes_file_path = 'data/hero_code_to_name.csv'

matches_df = pd.read_csv(matches_file_path)

matches_df.head()

,my_team_picks,enemy_team_picks,my_bans,enemy_bans,first_pick_team,winner
0,"[{""hero_code"": ""c2042"", ""pick_order"": 2}, {""he...","[{""hero_code"": ""c2007"", ""pick_order"": 1}, {""he...","[""c2066"", ""c2011""]","[""c2066"", ""c2113""]",enemy_team,my_team
1,"[{""hero_code"": ""c2112"", ""pick_order"": 2}, {""he...","[{""hero_code"": ""c4158"", ""pick_order"": 1}, {""he...","[""c2066"", ""c2011""]","[""c1153"", ""c1117""]",enemy_team,my_team
2,"[{""hero_code"": ""c2042"", ""pick_order"": 2}, {""he...","[{""hero_code"": ""c2090"", ""pick_order"": 1}, {""he...","[""c2066"", ""c2011""]","[""c2066"", ""c5110""]",enemy_team,my_team
3,"[{""hero_code"": ""c2112"", ""pick_order"": 1}, {""he...","[{""hero_code"": ""c2042"", ""pick_order"": 2}, {""he...","[""c2066"", ""c2011""]","[""c1153"", ""c1117""]",my_team,my_team
4,"[{""hero_code"": ""c2042"", ""pick_order"": 1}, {""he...","[{""hero_code"": ""c1153"", ""pick_order"": 2}, {""he...","[""c2066"", ""c2011""]","[""c5110"", ""c2112""]",my_team,enemy_team


Rearrange to winning and losing teams

In [70]:
# Iterate over each match and apply the rules
lst = []
for index, row in matches_df.iterrows():
    winner = row['winner']  # Adjust this if your column name is different
    
    # Map winner to my_team and loser to enemy_team
    first_pick_mapping = {'my_team':'winning_team', 'enemy_team':'losing_team'} if winner == 'my_team' else {'my_team':'losing_team', 'enemy_team':'winning_team'}
    mapping = {'winning_team':'my', 'losing_team':'enemy'} if winner == 'my_team' else {'winning_team':'enemy', 'losing_team':'my'}

    new_row = {
        'winning_team_picks': row[mapping['winning_team'] + '_team_picks'],
        'losing_team_picks': row[mapping['losing_team'] + '_team_picks'],
        'winning_team_bans': row[mapping['winning_team'] + '_bans'],
        'losing_team_bans': row[mapping['losing_team'] + '_bans'],
        'first_pick_team': first_pick_mapping[row['first_pick_team']] 
    }

    # Append the new row
    lst.append(new_row)

    
# Initialize a new DataFrame to store the rearranged data
rearranged_matches_df = pd.DataFrame(lst, columns=[
    'winning_team_picks',
    'losing_team_picks',
    'winning_team_bans',
    'losing_team_bans',
    'first_pick_team'
])

# Shuffle the matches
rearranged_matches_df = rearranged_matches_df.sample(frac=1).reset_index(drop=True)

# Save the new DataFrame to a CSV file
rearranged_matches_df.to_csv('data/rearranged_epic7_matches.csv', index=False)
print('Rearranged data has been saved to data/rearranged_epic7_matches.csv')

rearranged_matches_df.head()

Rearranged data has been saved to data/rearranged_epic7_matches.csv


,winning_team_picks,losing_team_picks,winning_team_bans,losing_team_bans,first_pick_team
0,"[{""hero_code"": ""c5110"", ""pick_order"": 1}, {""he...","[{""hero_code"": ""c2106"", ""pick_order"": 2}, {""he...","[""c1153"", ""c1133""]","[""c1153"", ""c2113""]",winning_team
1,"[{""hero_code"": ""c2112"", ""pick_order"": 2}, {""he...","[{""hero_code"": ""c5110"", ""pick_order"": 1}, {""he...","[""c2066"", ""c2090""]","[""c1153"", ""c1133""]",losing_team
2,"[{""hero_code"": ""c2066"", ""pick_order"": 1}, {""he...","[{""hero_code"": ""c5110"", ""pick_order"": 2}, {""he...","[""c1118"", ""c1151""]","[""c1133"", ""c4158""]",winning_team
3,"[{""hero_code"": ""c2011"", ""pick_order"": 1}, {""he...","[{""hero_code"": ""c5110"", ""pick_order"": 2}, {""he...","[""c2066"", ""c2112""]","[""c2066"", ""c1153""]",winning_team
4,"[{""hero_code"": ""c2042"", ""pick_order"": 1}, {""he...","[{""hero_code"": ""c2090"", ""pick_order"": 2}, {""he...","[""c2011"", ""c2066""]","[""c1153"", ""c2113""]",winning_team


convert the matches to snapshots and one hot encode

In [71]:
import numpy as np
import ast
import pandas as pd

# Load the hero codes mapping
hero_mapping_df = pd.read_csv(hero_codes_file_path, header=None, names=['code', 'name'])
hero_codes = hero_mapping_df['code'].tolist()

# Define the vocabulary
vocab = ['Null', 'Start', 'Ban', 'Pick', 'winning_team', 'losing_team', 'c_null'] + hero_codes + [str(i) for i in range(11)]
vocab_size = len(vocab)

# Create a mapping from word to index for one-hot encoding
word_to_index = {word: idx for idx, word in enumerate(vocab)}
index_to_word = {idx: word for word, idx in word_to_index.items()}

# Function to convert JSON-like strings into lists of dictionaries
def parse_json_like_string(json_str):
    try:
        return ast.literal_eval(json_str)
    except ValueError:
        return []

# Function to create snapshots for predicting the hero code
def create_pick_snapshots(match):
    snapshots = []
    targets = []
    sequence = []

    # Parse the JSON-like strings
    winning_picks = parse_json_like_string(match['winning_team_picks'])
    losing_picks = parse_json_like_string(match['losing_team_picks'])
    winning_bans = parse_json_like_string(match['winning_team_bans'])
    losing_bans = parse_json_like_string(match['losing_team_bans'])

    # Add all prebans to the sequence
    for ban in winning_bans:
        sequence.extend(['Ban', '0', 'winning_team', ban if ban else 'c_null'])
    for ban in losing_bans:
        sequence.extend(['Ban', '0', 'losing_team', ban if ban else 'c_null'])

    # Combine and sort all picks
    all_picks = winning_picks + losing_picks
    all_picks_sorted = sorted(all_picks, key=lambda x: x['pick_order'])

    # Create snapshots and targets for picks 1 to 10
    for pick in all_picks_sorted:
        pick_order = int(pick['pick_order'])
        team = 'winning_team' if pick in winning_picks else 'losing_team'

        # Only create snapshots for picks 1 to 10
        if 1 <= pick_order <= 10:
            # Create a snapshot before the current pick
            snapshot = sequence.copy()

            # Add the first 3 words of the target to the end of the snapshot
            snapshot.extend(['Pick', str(pick_order), team])

            # Ensure the snapshot is exactly 56 words long
            while len(snapshot) < 56:
                snapshot.append('Null')  # Pad to 56 words
            
            snapshots.append(snapshot)

            # Set the target to be only the hero code (4th word)
            hero_code = pick['hero_code']
            targets.append(hero_code)

        # Add the current pick to the sequence after creating the snapshot
        sequence.extend(['Pick', str(pick_order), team, pick['hero_code']])

    return snapshots, targets

# Convert all matches into snapshots and targets
results = rearranged_matches_df.apply(create_pick_snapshots, axis=1).tolist()

# Flatten the list of snapshots and targets
snapshots = [snapshot for match_snapshots, _ in results for snapshot in match_snapshots]
targets = [target for _, match_targets in results for target in match_targets]

# Convert the snapshots into one-hot encoded format
def one_hot_encode_sequence(sequence, vocab_size, word_to_index):
    one_hot_encoded = np.zeros((len(sequence), vocab_size), dtype=np.int32)
    for i, word in enumerate(sequence):
        word_index = word_to_index.get(word, word_to_index['Null'])
        one_hot_encoded[i, word_index] = 1
    return one_hot_encoded

# One-hot encode the entire snapshots
one_hot_encoded_snapshots = np.array([one_hot_encode_sequence(snapshot, vocab_size, word_to_index) for snapshot in snapshots])

# Convert the targets to one-hot encoded format
def one_hot_encode_target(target, vocab_size, word_to_index):
    one_hot_encoded = np.zeros(vocab_size, dtype=np.int32)
    one_hot_encoded[word_to_index.get(target, word_to_index['Null'])] = 1
    return one_hot_encoded

# One-hot encode the targets
one_hot_encoded_targets = np.array([one_hot_encode_target(target, vocab_size, word_to_index) for target in targets])

# Output the shape of the encoded data for verification
print(f'One-hot encoded snapshots shape: {one_hot_encoded_snapshots.shape}')
print(f'One-hot encoded targets shape: {one_hot_encoded_targets.shape}')


One-hot encoded snapshots shape: (99210, 56, 357)
One-hot encoded targets shape: (99210, 357)


Visualise the input and target data

In [72]:
# Function to decode one-hot encoded sequences back to words
def decode_one_hot_sequence(one_hot_encoded, index_to_word):
    return [index_to_word[np.argmax(token)] for token in one_hot_encoded]

# Format the decoded sequence into lines with 4 words each, excluding 'Null'
def format_decoded_sequence(decoded_sequence):
    
    # Remove 'Null' padding for easier reading
    filtered_sequence = [word for word in decoded_sequence if word != 'Null']
    lines = [' '.join(filtered_sequence[i:i+4]) for i in range(0, len(filtered_sequence), 4)]
    return '\n'.join(lines)

# print(decode_one_hot_sequence(one_hot_encoded_snapshots[0], index_to_word))

# Compare snapshots and targets with improved formatting
def compare_snapshots_and_targets(snapshots, targets, index_to_word, num_samples=5):
    for i in range(min(num_samples, len(snapshots))):
        # Decode the current snapshot
        decoded_snapshot = decode_one_hot_sequence(snapshots[i], index_to_word)
        
        # Decode the target (hero code)
        target_index = np.argmax(targets[i])  # Get the index of the predicted hero code
        decoded_target = index_to_word.get(target_index, 'Null')

        # Remove 'Null' padding and format the snapshot
        formatted_snapshot = format_decoded_sequence(decoded_snapshot)

        # Display the comparison with improved formatting
        print(f"Snapshot {i+1}:")
        print(formatted_snapshot)
        print("\nPredicted Hero Code:")
        print(decoded_target)
        print("=" * 50)

# Assuming 'one_hot_encoded_snapshots' and 'one_hot_encoded_targets' are available from earlier steps
# Call the comparison function to check the first 20 samples
compare_snapshots_and_targets(one_hot_encoded_snapshots, one_hot_encoded_targets, index_to_word, num_samples=20)


Snapshot 1:
Ban 0 winning_team c1153
Ban 0 winning_team c1133
Ban 0 losing_team c1153
Ban 0 losing_team c2113
Pick 1 winning_team

Predicted Hero Code:
c5110
Snapshot 2:
Ban 0 winning_team c1153
Ban 0 winning_team c1133
Ban 0 losing_team c1153
Ban 0 losing_team c2113
Pick 1 winning_team c5110
Pick 2 losing_team

Predicted Hero Code:
c2106
Snapshot 3:
Ban 0 winning_team c1153
Ban 0 winning_team c1133
Ban 0 losing_team c1153
Ban 0 losing_team c2113
Pick 1 winning_team c5110
Pick 2 losing_team c2106
Pick 3 losing_team

Predicted Hero Code:
c2016
Snapshot 4:
Ban 0 winning_team c1153
Ban 0 winning_team c1133
Ban 0 losing_team c1153
Ban 0 losing_team c2113
Pick 1 winning_team c5110
Pick 2 losing_team c2106
Pick 3 losing_team c2016
Pick 4 winning_team

Predicted Hero Code:
c2042
Snapshot 5:
Ban 0 winning_team c1153
Ban 0 winning_team c1133
Ban 0 losing_team c1153
Ban 0 losing_team c2113
Pick 1 winning_team c5110
Pick 2 losing_team c2106
Pick 3 losing_team c2016
Pick 4 winning_team c2042
Pick 

In [73]:
from sklearn.model_selection import train_test_split

# Split the data into training and temp sets (70% train, 30% temp)
X_train, X_temp, y_train, y_temp = train_test_split(
    one_hot_encoded_snapshots, one_hot_encoded_targets, 
    test_size=0.3, random_state=42
)

# Split the temp set into validation and test sets (15% each)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, 
    test_size=0.5, random_state=42
)

# Output the shapes of the split data
print(f"Training set shape: {X_train.shape}, {y_train.shape}")
print(f"Validation set shape: {X_val.shape}, {y_val.shape}")
print(f"Test set shape: {X_test.shape}, {y_test.shape}")

Training set shape: (69447, 56, 357), (69447, 357)
Validation set shape: (14881, 56, 357), (14881, 357)
Test set shape: (14882, 56, 357), (14882, 357)


In [74]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Define the LSTM model
model = Sequential()

# LSTM layer to process the 56-word input sequence
model.add(LSTM(128, input_shape=(56, 357), return_sequences=False))

# Dense layer to predict the hero code (single word)
model.add(Dense(357, activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Print model summary
model.summary()

C:\Users\User\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_10 (LSTM)                  │ (None, 128)            │       248,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 357)            │        46,053 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 294,885 (1.12 MB)

 Trainable params: 294,885 (1.12 MB)

 Non-trainable params: 0 (0.00 B)

In [75]:
# Train the model
history = model.fit(
    X_train, y_train, 
    validation_data=(X_val, y_val), 
    epochs=10, batch_size=32
)

# Plot the training and validation accuracy
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

Epoch 1/10
2171/2171 ━━━━━━━━━━━━━━━━━━━━ 43s 19ms/step - accuracy: 0.0588 - loss: 4.1699 - val_accuracy: 0.0866 - val_loss: 3.7721
Epoch 2/10
 602/2171 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - accuracy: 0.0895 - loss: 3.7576

KeyboardInterrupt: 

In [68]:
import numpy as np

# Make predictions on the test data
y_pred = model.predict(X_test)

# Function to decode one-hot encoded hero code back to words
def decode_hero_code(one_hot_vector, index_to_word):
    hero_code_index = np.argmax(one_hot_vector)  # Find the index of the predicted word
    return index_to_word.get(hero_code_index, 'Unknown')

# Visualise predictions and actual targets for a few examples
def visualise_single_word_predictions(X_test, y_test, y_pred, index_to_word, num_samples=5):
    for i in range(min(num_samples, len(X_test))):
        
        # Decode the input snapshot
        decoded_input = decode_one_hot_sequence(X_test[i], index_to_word)
        formatted_input = ' '.join([word for word in decoded_input if word != 'Null'])

        # Decode the actual target (hero code)
        actual_hero = decode_hero_code(y_test[i], index_to_word)

        # Decode the predicted target (hero code)
        predicted_hero = decode_hero_code(y_pred[i], index_to_word)

        # Display the input, actual target, and predicted target
        print(f"Example {i+1}:")
        print(f"Input Snapshot:\n{formatted_input}")
        print(f"Actual Hero Code: {actual_hero}")
        print(f"Predicted Hero Code: {predicted_hero}")
        print("=" * 50)

# Call the function to visualise the predictions on the test set
visualise_single_word_predictions(X_test, y_test, y_pred, index_to_word, num_samples=5)


47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
Example 1:
Input Snapshot:
Ban 0 winning_team c1153 Ban 0 winning_team c2066 Ban 0 losing_team c2066 Ban 0 losing_team c2090 Pick 1 losing_team c2112 Pick 2 winning_team c2042 Pick 3 winning_team c2089 Pick 4 losing_team c5110 Pick 5 losing_team c2113 Pick 6 winning_team c2011 Pick 7 winning_team c1151 Pick 8 losing_team c1125 Pick 9 losing_team
Actual Hero Code: c2088
Predicted Hero Code: c1151
Example 2:
Input Snapshot:
Ban 0 winning_team c2112 Ban 0 winning_team c1153 Ban 0 losing_team c1133 Ban 0 losing_team c1153 Pick 1 losing_team
Actual Hero Code: c2011
Predicted Hero Code: c2090
Example 3:
Input Snapshot:
Ban 0 winning_team c1153 Ban 0 winning_team c2112 Ban 0 losing_team c2066 Ban 0 losing_team c2112 Pick 1 losing_team c2042 Pick 2 winning_team c5110 Pick 3 winning_team c2090 Pick 4 losing_team c2016 Pick 5 losing_team c6037 Pick 6 winning_team c1159 Pick 7 winning_team c2113 Pick 8 losing_team c2088 Pick 9 losing_team
Actual Hero Code: 